# 05 · Calculate area against independent spherical and geodesic areas

**Question.** Does Fieldwork's `measure_area` widget compute the spherical area it
declares, and how far is that declared method from a geodesic area on an ellipsoid?

Those are two questions and this notebook keeps them apart. Collapsing them would report
a methodological choice as a defect.

1. **Implementation.** The widget declares "Turf 7.3.5 spherical polygon area; mean Earth
   radius 6371008.8 m", with the boundary following parallels and meridians. The check
   recomputes that from a different closed-form expression of the same quantity. Criterion
   fixed before running: relative difference at most 1e-9.
2. **Edge model.** A geodesic polygon area joins the same vertices with great circles. The
   difference is a modelling choice, reported and never failed.
3. **Earth model.** A WGS84 ellipsoidal area quantifies the approximation the widget
   already discloses when it calls its result approximate.

Nothing here imports Fieldwork code; the fixture carries the declared parameters and the
widget's own output, and its SHA-256 is verified on load.


In [ ]:
# pyproj ships in the Pyodide distribution, not as a PyPI wheel on this site, so it loads
# with pyodide_js.loadPackage. piplite.install would look in the site's local wheel index
# and fail. Outside Pyodide this cell does nothing.
NEEDED = ["pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_05_area import run

report = run(Path.cwd() / "fixtures" / "measure-area-001.json")
print("criterion:", report["criterion"])
print("agrees:", report["agrees"])
print(report["fixture"]["declaredMethod"])


## The three comparisons side by side

Implementation agreement should be near machine precision, because it is the same formula
computed differently. The other two columns are magnitudes, not verdicts.


In [ ]:
# Three columns, three different questions. The first is the only pass/fail one.
print(f"{'case':<24}{'implementation':>16}{'edge model':>14}{'earth model':>14}")
for case in report["cases"]:
    c = case["context"]
    print(f"{case['id']:<24}"
          f"{case['relativeDifference']:>16.2e}"
          f"{c['relativeToGreatCircleEdges']*100:>13.4f}%"
          f"{c['relativeToEllipsoidal']*100:>13.4f}%")


## The same differences in reporting units

A percentage is easy to wave away. The cell below restates the ellipsoidal difference in
square kilometres, which is the number that would appear in a report.


In [ ]:
# What the differences mean in the units a report would quote.
for case in report["cases"]:
    c = case["context"]
    km2 = case["fieldworkSquareMetres"] / 1e6
    gap = (case["fieldworkSquareMetres"] - c["wgs84EllipsoidalSquareMetres"]) / 1e6
    print(f"{case['id']:<24}{km2:>12.3f} km2   against WGS84: {gap:+.3f} km2")


## Reading the result

**The implementation agrees** to roughly 1e-14 relative, far inside the stated 1e-9. The
widget computes the spherical formula it says it computes.

**The edge model matters only for oblique edges.** Latitude–longitude boxes differ from
great-circle-edged areas by less than 0.01 %, because their edges already follow parallels
and meridians. The mid-latitude triangle, whose edges are long and oblique, differs by
about **0.3 %**.

**The earth model matters everywhere, and not by a constant.** Against WGS84 the spherical
area runs about **+0.45 % at the equator** and about **−0.56 % at 60° N**: the sign changes
with latitude, so it cannot be corrected by a single factor.

None of this makes the widget wrong — it declares a spherical approximation. What the check
adds is the size of that approximation, measured, so a practitioner can decide whether
0.2–0.6 % matters for their purpose. For a denominator in a rate, usually not. For a
boundary dispute or an area-based payment, possibly.
